<h3>Statements</h3>
Take a look at this public BigQuery Dataset of stackoverflow posts. <br>
Please provide queries which answer the following prompts <br>
1. What tags on a Stack Overflow question lead to the most answers and the highest rate 
of approved answers for the current year? What tags lead to the least? How about 
combinations of tags? <br>
2. For posts which are tagged with only ‘python’ or ‘dbt’, what is the year over year change 
of question-to-answer ratio for the last 10 years? How about the rate of approved 
answers on questions for the same time period? How do posts tagged with only ‘python’ 
compare to posts only tagged with ‘dbt’? <br>
3. Other than tags, what qualities on a post correlate with the highest rate of answer and 
approved answer? Feel free to get creative

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from google.cloud import bigquery

import warnings  
warnings.filterwarnings("ignore")

<h3>Data Retrieval and basic transformations</h3>

In [2]:
'''
# Retrieve the data directly from BigQuery
client = bigquery.Client(project= "stackoverflow-challenge")

query = """
SELECT
  id,
  creation_date,
  tags,
  answer_count,
  comment_count,
  accepted_answer_id,
  score,
  view_count,
  favorite_count,
  owner_display_name,
  owner_user_id,
FROM `bigquery-public-data.stackoverflow.posts_questions`
WHERE creation_date >= TIMESTAMP_SUB(
    (SELECT MAX(creation_date)
     FROM `bigquery-public-data.stackoverflow.posts_questions`),
    INTERVAL 3650 DAY
)
"""

df = client.query(query).to_dataframe(create_bqstorage_client=True)

# Save it to parquet (so it doesn't has to be downloaded again if needed)
df.to_parquet("C:/Users/aaron/Downloads/stackoverflow2.parquet", index = False)


# most recent date = 2022-09-25 05:56:36.210000 UTC
'''

'\n# Retrieve the data directly from BigQuery\nclient = bigquery.Client(project= "stackoverflow-challenge")\n\nquery = """\nSELECT\n  id,\n  creation_date,\n  tags,\n  answer_count,\n  comment_count,\n  accepted_answer_id,\n  score,\n  view_count,\n  favorite_count,\n  owner_display_name,\n  owner_user_id,\nFROM `bigquery-public-data.stackoverflow.posts_questions`\nWHERE creation_date >= TIMESTAMP_SUB(\n    (SELECT MAX(creation_date)\n     FROM `bigquery-public-data.stackoverflow.posts_questions`),\n    INTERVAL 3650 DAY\n)\n"""\n\ndf = client.query(query).to_dataframe(create_bqstorage_client=True)\n\n# Save it to parquet (so it doesn\'t has to be downloaded again if needed)\ndf.to_parquet("C:/Users/aaron/Downloads/stackoverflow2.parquet", index = False)\n\n\n# most recent date = 2022-09-25 05:56:36.210000 UTC\n'

In [3]:
df = pd.read_parquet(r"C:\Users\aaron\Downloads\stackoverflow2.parquet")

In [4]:
df['creation_date'] = pd.to_datetime(df['creation_date'])

# Transform accepted_answer_id to boolean
df["accepted_answer_id"] = df["accepted_answer_id"].notna().astype(int)

# Fill the missing values from the column "owner_user_id" with the values from "owner_display_name"
df["owner_user_id"] = (
                        df["owner_user_id"]
                        .astype("string")
                        .fillna(df["owner_display_name"])
                )

df = df.drop(columns=["owner_display_name"])

In [5]:
df.describe()

,id,answer_count,comment_count,accepted_answer_id,score,view_count,favorite_count
count,19562489.0,19562489.0,19562489.0,1.956249e+07,19562489.0,19562489.0,3714604.0
mean,43783314.004128,1.327178,2.10116,4.788092e-01,1.420311,1878.381216,1.922185
std,17754274.016048,1.18207,2.732885,4.995508e-01,10.544411,13344.592824,8.20095
min,12614937.0,0.0,0.0,0.000000e+00,-138.0,1.0,0.0
25%,28571646.0,1.0,0.0,0.000000e+00,0.0,83.0,1.0
50%,43786137.0,1.0,1.0,0.000000e+00,0.0,282.0,1.0
75%,59162047.0,2.0,3.0,1.000000e+00,1.0,1011.0,2.0
max,73842327.0,117.0,108.0,1.000000e+00,6894.0,9378947.0,5889.0


<h3> Problems Solving </h3>

<h4>1-a. What tags on a Stack Overflow question lead to the most answers and the highest rate 
of approved answers for the current year?</h4>

In [6]:
# Filter to stay only with the last 365 days of data
max_date = df["creation_date"].max()
tags_df = df[df["creation_date"] >= max_date - pd.Timedelta(days=365)]

# Create a copy of this df to use later
tags_df2 = tags_df.copy()

# Select only the needed columns (make the df lighter)
tags_df = tags_df[["id", "tags", "answer_count", "accepted_answer_id"]]

In [7]:
# Split the tags into new columns and then explode it into new rows
tags_df["tags"] = tags_df["tags"].str.split("|")
tags_df = tags_df.explode("tags")

# Do aggregations as requested
tags_df = (
    tags_df.groupby("tags").agg(
                                answers_sum_total  = ("answer_count", "sum"),         # Total  number of answers received per tag
                                answers_avg        = ("answer_count", "mean"),        # Average number of answers per tag (each post has that number of answers in average)
                                accepted_rate      = ("accepted_answer_id", "mean"),  # The percentage of accepted answers (1's)(rate)
                                questions_count    = ("id", "count"),                 # Total number of questions per tag
                            ).reset_index()
        )

As good practice, the number of questions asked per tag (questions_count column) was filtered over 100 to avoid having tags with basically no questions; as it can be misleading of the results.

In [8]:
tags_df = tags_df[tags_df["questions_count"] > 100]

In [9]:
# Most answered tags in average (as a rate)
tags_df.sort_values("answers_avg", ascending=False).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
3740,awk,4122,2.382659,0.661272,1730
35302,sed,3133,2.181755,0.619777,1436
16721,gsub,322,1.951515,0.654545,165
14609,function-definition,640,1.887906,0.504425,339
5819,c-strings,760,1.809524,0.473810,420
16570,grep,1551,1.793064,0.538728,865
38247,string-literals,209,1.741667,0.533333,120
38274,stringr,643,1.737838,0.708108,370
10103,dictionary-comprehension,223,1.715385,0.538462,130
22212,list-comprehension,1173,1.682927,0.572453,697


In [10]:
# Most answered tags in total (the total sum)
tags_df.sort_values("answers_sum_total", ascending=False).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
31243,python,262817,0.956397,0.360154,274799
19762,javascript,181140,0.954408,0.343211,189793
32964,reactjs,87813,0.87713,0.304393,100114
19623,java,72972,0.810143,0.281749,90073
17576,html,71427,1.033975,0.349059,69080
5756,c#,63595,0.868998,0.338307,73182
28461,pandas,56599,1.173036,0.504124,48250
32084,r,55977,1.042189,0.461488,53711
8583,css,52569,1.090462,0.362243,48208
37464,sql,47262,1.122053,0.415185,42121


In [11]:
# Highest rate of approved answers
tags_df.sort_values("accepted_rate", ascending=False).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
16181,google-query-language,409,1.21365,0.890208,337
13819,flatten,417,1.252252,0.759760,333
32288,raku,280,1.497326,0.721925,187
44938,z3py,128,0.969697,0.719697,132
20829,kdb,347,1.591743,0.711009,218
38274,stringr,643,1.737838,0.708108,370
30854,purrr,739,1.4294,0.692456,517
9230,data.table,1509,1.586751,0.681388,951
40047,tidyr,859,1.517668,0.680212,566
20275,jq,1639,1.469955,0.678027,1115


In [12]:
# EXTRA
# Top 100 most answered tags with the highest rate of approved answers
tags_df.sort_values("answers_avg", ascending=False).head(100).sort_values("accepted_rate", ascending=False).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
32288,raku,280,1.497326,0.721925,187
20829,kdb,347,1.591743,0.711009,218
38274,stringr,643,1.737838,0.708108,370
30854,purrr,739,1.4294,0.692456,517
9230,data.table,1509,1.586751,0.681388,951
40047,tidyr,859,1.517668,0.680212,566
20275,jq,1639,1.469955,0.678027,1115
3740,awk,4122,2.382659,0.661272,1730
11038,dplyr,8357,1.509301,0.657396,5537
16721,gsub,322,1.951515,0.654545,165


<h4>1-b. What tags lead to the least?</h4>

In [13]:
# Less answered tags in average (as a rate)
tags_df.sort_values("answers_avg", ascending=True).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
1167,amp-html,32,0.275862,0.094828,116
36265,simulink,66,0.280851,0.106383,235
6741,ckeditor5,75,0.298805,0.087649,251
11565,eclipse-cdt,32,0.299065,0.112150,107
9053,darknet,33,0.302752,0.073394,109
22114,linkedin-api,75,0.304878,0.060976,246
18088,icloud,34,0.306306,0.099099,111
12951,facebook-graph-api,228,0.31105,0.103683,733
5501,browser-extension,35,0.3125,0.098214,112
32506,rdlc,46,0.315068,0.089041,146


In [14]:
# Less answered tags in total (the total sum)
tags_df.sort_values("answers_sum_total", ascending=True).head(10)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
11565,eclipse-cdt,32,0.299065,0.112150,107
1167,amp-html,32,0.275862,0.094828,116
9053,darknet,33,0.302752,0.073394,109
18088,icloud,34,0.306306,0.099099,111
23573,memcached,34,0.326923,0.134615,104
5501,browser-extension,35,0.3125,0.098214,112
12964,facebook-javascript-sdk,35,0.318182,0.100000,110
169,aar,36,0.324324,0.081081,111
2677,applepay,36,0.324324,0.099099,111
13561,fingerprint,37,0.366337,0.099010,101


In [15]:
del tags_df

<h4>1-c. How about combinations of tags? </h4>

In [16]:
# No split/explode, just groupby by the existing tags (no new combinatios were created)
tags_df2 = (
    tags_df2.groupby("tags").agg(
                        answers_sum_total  = ("answer_count", "sum"),
                        answers_avg        = ("answer_count", "mean"),
                        accepted_rate      = ("accepted_answer_id", "mean"),
                        questions_count    = ("id", "count"),  
                    )
    .reset_index()
)

tags_df2 = tags_df2[tags_df2["questions_count"] > 100]
tags_df2.sort_values("answers_avg", ascending=False).head(10)

# awk is also the most popular topic by itself (above analytics)

,tags,answers_sum_total,answers_avg,accepted_rate,questions_count
82264,awk,576,2.341463,0.723577,246
94316,bash|awk,272,2.305085,0.644068,118
687232,python|string|list,319,2.155405,0.601351,148
629718,python|list|dictionary,872,2.046948,0.535211,426
629260,python|list-comprehension,220,2.037037,0.546296,108
667763,python|python-3.x|list,622,2.019481,0.529221,308
314703,javascript|arrays|object,809,1.963592,0.563107,412
96407,bash|sed,221,1.955752,0.610619,113
665713,python|python-3.x|dictionary,326,1.952096,0.646707,167
314058,javascript|arrays|javascript-objects,193,1.910891,0.574257,101


In [17]:
del tags_df2

<h4>2-a For posts which are tagged with only ‘python’ or ‘dbt’, what is the year over year change of question-to-answer ratio for the last 10 years? </h4>

In [18]:
tags_df3 = df[["creation_date", "id", "tags", "answer_count", "accepted_answer_id"]]

# Select only the desired tags
tags_df3 = tags_df3[((tags_df3["tags"] == "python") | (tags_df3["tags"] == "dbt"))]

# Extract the year of the creation_date into a new column
tags_df3["year"] = tags_df3["creation_date"].dt.year

# Groupby to obtain the number of questions and answers
yearly_ratio = (tags_df3.groupby(["year", "tags"]).agg(
                                                        questions_count        = ("id", "count"),  
                                                        answers_sum_total      = ("answer_count", "sum"),
                                                        answers_accepted_sum   = ("accepted_answer_id", "sum"),
                                                    )
                        .reset_index()
            )

# Create a copy to use later
yearly_ratio_original = yearly_ratio.copy()

del tags_df3

In [19]:
# Obtain the ratio (number of answers per each question)
yearly_ratio["ratio_qa_ans"] = (yearly_ratio["answers_sum_total"] / yearly_ratio["questions_count"]).round(1)

# Obtain the difference per year per each tag
yearly_ratio["year_difference"] = (yearly_ratio.groupby("tags")
                                    ["ratio_qa_ans"].diff()
                                    ).round(2)

# Obtain the porcentage change per year per each tag
yearly_ratio["year_pct_change"] = (yearly_ratio.groupby("tags")
                               ["ratio_qa_ans"].pct_change()
                               ).round(2)

yearly_ratio.sort_values(by = ["tags", "year"], ascending = True)

,year,tags,questions_count,answers_sum_total,answers_accepted_sum,ratio_qa_ans,year_difference,year_pct_change
8,2020,dbt,31,43,13,1.4,<NA>,<NA>
10,2021,dbt,58,66,15,1.1,-0.3,-0.21
12,2022,dbt,79,84,22,1.1,0.0,0.0
0,2012,python,1766,4266,1225,2.4,<NA>,<NA>
1,2013,python,7819,18068,5096,2.3,-0.1,-0.04
2,2014,python,8537,16792,5247,2.0,-0.3,-0.13
3,2015,python,9781,18582,5565,1.9,-0.1,-0.05
4,2016,python,10344,18970,5433,1.8,-0.1,-0.05
5,2017,python,11683,19851,5740,1.7,-0.1,-0.06
6,2018,python,11614,19653,5714,1.7,0.0,0.0


In [20]:
del yearly_ratio

<h4>2-b How about the rate of approved answers on questions for the same time period?</h4>

In [21]:
# Obtain the ratio (number of approved answers per each question)
yearly_ratio_original["ratio_approved_qa_ans"] = (yearly_ratio_original["answers_accepted_sum"] / yearly_ratio_original["questions_count"]).round(1)

# Obtain the difference per year per each tag
yearly_ratio_original["year_difference_approved"] = (yearly_ratio_original.groupby("tags")
                                                    ["ratio_approved_qa_ans"].diff()
                                                    ).round(2)

# Obtain the porcentage change per year per each tag
yearly_ratio_original["year_pct_change_approved"] = (yearly_ratio_original.groupby("tags")
                                                    ["ratio_approved_qa_ans"].pct_change()
                                                    ).round(2)

yearly_ratio_original.sort_values(by = ["tags", "year"], ascending = True)

,year,tags,questions_count,answers_sum_total,answers_accepted_sum,ratio_approved_qa_ans,year_difference_approved,year_pct_change_approved
8,2020,dbt,31,43,13,0.4,<NA>,<NA>
10,2021,dbt,58,66,15,0.3,-0.1,-0.25
12,2022,dbt,79,84,22,0.3,0.0,0.0
0,2012,python,1766,4266,1225,0.7,<NA>,<NA>
1,2013,python,7819,18068,5096,0.7,0.0,0.0
2,2014,python,8537,16792,5247,0.6,-0.1,-0.14
3,2015,python,9781,18582,5565,0.6,0.0,0.0
4,2016,python,10344,18970,5433,0.5,-0.1,-0.17
5,2017,python,11683,19851,5740,0.5,0.0,0.0
6,2018,python,11614,19653,5714,0.5,0.0,0.0


In [22]:
del yearly_ratio_original

<h4>2-c How do posts tagged with only ‘python’ compare to posts only tagged with ‘dbt’? </h4>

For Python, the number of questions increased from 2012 to 2020, then it decreased. However, the number of answers had low variance from 2013 to 2021. <br>
Regarding the ratio of questions/answers, the ratio was 2.4 at 2012 and has been decreasing since then. Every year the ratio has decreased or stayed the same.<br>
Regarding the questions/approved answers, the difference between years is minimal, but with a negative trend.<br>
More questions but basically the same number of answers <br>
<br>
dbt is much newer and there are records only for three years, both the number of questions and answers have increased, but is less popular than python by a factor of 100-150. <br>
Regarding the ratio of questions/answers, it has been stable with negative differences.<br>
Regarding the questions/approved answers, the difference between years is minimal, but with a negative trend.<br>
<br>
Overall, we can assure that based on the number of answers and accepted answers metrics, the engagment hasn't gone up with neither of the tags, at the contrary, it has decreased. <br>
This could be related to the tags/tools/languages or it could be an overall indicator of the plastform usage.
<br>
- Please note that 2022 has no complete data.

<h4>3. Other than tags, what qualities on a post correlate with the highest rate of answer and 
approved answer? Feel free to get creative</h4>

In [23]:
numeric_df = df[["creation_date","answer_count", "comment_count", 
                 "accepted_answer_id", "score", "view_count", 
                 'favorite_count']]


In [24]:
# Feature engineering to create new features to correlate
numeric_df["year"] = numeric_df["creation_date"].dt.year
numeric_df["month"] = numeric_df["creation_date"].dt.month
numeric_df["day_of_week"] = numeric_df["creation_date"].dt.dayofweek
numeric_df["hour"] = numeric_df["creation_date"].dt.hour
numeric_df = numeric_df.drop("creation_date", axis = 1)

In [25]:
numeric_df_corr = numeric_df.corr()

fig = go.Figure(go.Heatmap(
    z=numeric_df_corr.values,
    x=numeric_df_corr.columns.tolist(),
    y=numeric_df_corr.columns.tolist(),
    text=numeric_df_corr.round(2).values,
    texttemplate="%{text}",
    colorscale="RdBu",
    zmin=-1,
    zmax=1,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title = "Correlation between numeric columns",
    xaxis=dict(tickangle=-45),
    width=800,
    height=700
)

fig.show()

# Check the correlation between answer_count and accepted_answer_id vs all the others

The following analysis is to analyze the Number of Answers

In [26]:
# Plot Number of Answers vs Score
# As the data is to heavy split the data into bins to plot
numeric_df["score_bin"] = pd.cut(numeric_df["score"], bins=60)
# Group to obtain the mean of answers per each bin (the sum makes no sense)
grouped = numeric_df.groupby("score_bin")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["score_bin"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Score Range",
    xaxis_title="Score Range (bins)",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [27]:
# Plot Number of Answers vs Number of Views
# As the data is to heavy split the data into bins to plot
numeric_df["view_count_bin"] = pd.cut(numeric_df["view_count"], bins=60)
# Group to obtain the mean of answers per each bin (the sum makes no sense)
grouped = numeric_df.groupby("view_count_bin")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["view_count_bin"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per View Counts Range(bins)",
    xaxis_title="View Counts Range (bins)",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [28]:
# Plot Number of Answers vs Number of Favorites
# As the data is to heavy split the data into bins to plot
numeric_df["favorite_count_bin"] = pd.cut(numeric_df["favorite_count"], bins=60)
# Group to obtain the mean of answers per each bin (the sum makes no sense)
grouped = numeric_df.groupby("favorite_count_bin")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["favorite_count_bin"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Favorite Count",
    xaxis_title="Favorite Counts Range (bins)",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [29]:
# Plot Number of Answers vs Year
# As the data is to heavy groupby year to plot
grouped = numeric_df.groupby("year")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["year"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Year",
    xaxis_title="Year",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [30]:
# Plot Number of Answers vs Month of the Year
# As the data is to heavy groupby year to plot
grouped = numeric_df.groupby("month")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["month"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Month",
    xaxis_title="Month",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [31]:
# Plot Number of Answers vs Hours it was posted
# As the data is to heavy groupby year to plot
grouped = numeric_df.groupby("hour")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["hour"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Hour",
    xaxis_title="Hour",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [32]:
del numeric_df
del grouped

The Score Range and the View Counts are positively correlated to the mean number of answers, its not a perfect correlation but there's a visible trend.
This means that the posts that have more views or have a higher score tend to have a higher number of answers. <br>
The Year has a negative correlation, meaning that the most recent years have less answers in average. <br>
Month of the year and hour of day have no correlation or trend with the number of answers. The favorite count is uncertain/unclear.

The following analysis is to analyze the Accepted Answers

In [33]:
tags_df = df[["id", "tags", "answer_count", "accepted_answer_id"]]

# Split the tags into new columns and then explode it into new rows
#tags_df["tags"] = tags_df["tags"].str.split("|")
#tags_df = tags_df.explode("tags")

# A groupby is needed because for the accepted answers we only have 1 or 0 and the sum/average must me calculated first
# Do aggregations
tags_df = (
    tags_df.groupby("tags").agg(
                                answers_sum_total  = ("answer_count", "sum"),         # Total answers received per tag
                                answers_avg        = ("answer_count", "mean"),        # Average number of questions per tag
                                accepted_rate      = ("accepted_answer_id", "mean"),  # The percentage of accepted answers (1's)(rate)
                                questions_count    = ("id", "count"),                 # Total questions per tag
                            ).reset_index()
        )

In [34]:
tags_df = tags_df[["answers_sum_total","answers_avg","accepted_rate","questions_count"]]
tags_df = tags_df[tags_df["questions_count"] > 100]
tags_df_corr = tags_df.corr()

fig = go.Figure(go.Heatmap(
    z=tags_df_corr.values,
    x=tags_df_corr.columns.tolist(),
    y=tags_df_corr.columns.tolist(),
    text=tags_df_corr.round(2).values,
    texttemplate="%{text}",
    colorscale="RdBu",
    zmin=-1,
    zmax=1,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    xaxis=dict(tickangle=-45),
    width=800,
    height=700
)

fig.show()

# Accepted rate (the most relevant for this analysis) shows correlation to the average number of answers

In [35]:
# As the data is to heavy split the data into bins to plot
tags_df["answers_avg_bin"] = pd.cut(tags_df["answers_avg"], bins=60)
# Group to obtain the mean of answers per each bin (the sum makes no sense)
grouped = tags_df.groupby("answers_avg_bin")["accepted_rate"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["answers_avg_bin"].astype(str),
    y=grouped["accepted_rate"]
))

fig.update_layout(
    title="Average Number of Answers vs Accepted Rate",
    xaxis_title="Average Number of Answers",
    yaxis_title="Accepted Rate per Post"
)
fig.show()

In [36]:
del tags_df
del grouped

The answer accepted rate its highly correlated to the average number of answers. The more answers the more common is to have an accepted one.

<h4 >EXTRA!! <br> Final analysis, Are there "StackOverflow" influencers? Is having more posts make each post have more answers or accepted answers? </h4>

In [37]:
df["owner_user_id"].nunique()
# Over 4.3M authors

4387881

In [38]:
# Obtain the number of posts per each owner/author
df_owners = pd.DataFrame(df["owner_user_id"].value_counts()).sort_values("count", ascending=False)

# To reduce the df size, filter to stay with those owners with more than 5 posts
df_owners = df_owners[df_owners["count"] > 5]

df_owners = df_owners.reset_index()

# Filter the original df to make sure it only has the authors from "df_owners" (to reduce the size)
df2 = df[df["owner_user_id"].isin(df_owners["owner_user_id"])].drop_duplicates()

# Do a left join to merge the recent created dfs
df2 = df2.merge(df_owners, on = "owner_user_id", how = "left")
df.head()

del df_owners


In [39]:
df2 = df2[["answer_count", "accepted_answer_id", "score", "view_count", "count"]]
df2_corr = df2.corr()

fig = go.Figure(go.Heatmap(
    z=df2_corr.values,
    x=df2_corr.columns.tolist(),
    y=df2_corr.columns.tolist(),
    text=df2_corr.round(2).values,
    texttemplate="%{text}",
    colorscale="RdBu",
    zmin=-1,
    zmax=1,
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title = "Correlation between numeric columns",
    xaxis=dict(tickangle=-45),
    width=800,
    height=700
)

fig.show()

del df2_corr

# The Count colums (the most relevant for this analysis) shows no correlation with others

In [40]:
# Plot the Number of Posts per Author vs Number of Answers
# As the data is to heavy split the data into bins to plot
df2["count_bin"] = pd.cut(df2["count"], bins=30)
# Group to obtain the mean of answers per each bin (the sum makes no sense)
grouped = df2.groupby("count_bin")["answer_count"].mean().reset_index()
grouped = grouped.dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=grouped["count_bin"].astype(str),
    y=grouped["answer_count"]
))

fig.update_layout(
    title="Average Number of Answers per Number of Posts",
    xaxis_title="Number of Posts (bins)",
    yaxis_title="Average Number of Answers"
)
fig.show()

In [41]:
# Also, just in case, obtain the correlation between the number of posts per author and the accepted answers mean
df2 = df.groupby("owner_user_id").agg(
    total_posts=("id", "count"),
    total_answers=("accepted_answer_id", "mean")
)

df2.corr()

,total_posts,total_answers
total_posts,1.00000,0.10264
total_answers,0.10264,1.00000


In [42]:
del df2
del grouped

An author having many posts doesn't mean that he will have more answers, neither accepted answers per post. <br>
So, no, there are no "influencers".